# Credit Scoring Model
Predicting individual creditworthiness from financial history using classification models.

**Dataset:** This notebook is built for the *"Give Me Some Credit"* dataset from Kaggle
(https://www.kaggle.com/c/GiveMeSomeCredit/data), a widely used credit-scoring dataset with
income, debt ratio, and payment-history features. If you use a different Kaggle credit dataset,
just update the `DATA_PATH` and column names in the **Configuration** cell below — the rest of
the pipeline is written to be adaptable to any binary "default / no default" style dataset.

**Pipeline:**
1. Load & inspect data
2. Exploratory Data Analysis (EDA)
3. Feature engineering
4. Preprocessing (missing values, scaling)
5. Train/test split
6. Train Logistic Regression, Decision Tree, Random Forest
7. Evaluate with Precision, Recall, F1-Score, ROC-AUC
8. Compare models & save the best one


## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)
import joblib

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
RANDOM_STATE = 42


## 2. Configuration

Edit these two things after downloading the dataset:
- `DATA_PATH`: path to the CSV you downloaded from Kaggle
- `TARGET_COL`: the name of the target/label column (1 = defaulted / high risk, 0 = good credit)

**To download the "Give Me Some Credit" dataset via the Kaggle API:**
```bash
kaggle competitions download -c GiveMeSomeCredit -p ../data
cd ../data && unzip GiveMeSomeCredit.zip
```
This produces `cs-training.csv`, which has a `SeriousDlqin2yrs` target column (matches the default below).
If you instead grabbed a different Kaggle credit dataset, just change `TARGET_COL` to whatever the
label column is called there (e.g. `default`, `Creditworthy`, `Risk`, etc.).


In [ ]:
DATA_PATH = '../data/cs-training.csv'   # <-- update to your downloaded file
TARGET_COL = 'SeriousDlqin2yrs'          # <-- update to your target column name


## 3. Load Data

In [ ]:
df = pd.read_csv(DATA_PATH)

# Drop stray index column some Kaggle exports include
if df.columns[0].lower() in ('unnamed: 0', 'id'):
    df = df.drop(columns=[df.columns[0]])

print(f"Shape: {df.shape}")
df.head()


In [ ]:
df.info()


In [ ]:
df.describe().T


## 4. Exploratory Data Analysis

Understand class balance, missing values, and how features relate to creditworthiness.


In [ ]:
print("Missing values per column:")
print(df.isnull().sum().sort_values(ascending=False))


In [ ]:
target_counts = df[TARGET_COL].value_counts(normalize=True) * 100
print(target_counts)

plt.figure()
sns.countplot(x=TARGET_COL, data=df)
plt.title('Class Distribution (0 = Good Credit, 1 = High Risk / Default)')
plt.show()


In [ ]:
# Correlation heatmap (numeric features only)
plt.figure(figsize=(10, 8))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), cmap='coolwarm', center=0, annot=False)
plt.title('Feature Correlation Heatmap')
plt.show()


In [ ]:
# Distribution of a key feature (income) split by target, if present
income_col = [c for c in df.columns if 'income' in c.lower()]
if income_col:
    col = income_col[0]
    plt.figure()
    sns.histplot(data=df, x=col, hue=TARGET_COL, bins=50, log_scale=(False, True))
    plt.xlim(0, df[col].quantile(0.99))
    plt.title(f'{col} distribution by target')
    plt.show()


## 5. Feature Engineering

Build derived features that tend to matter for credit risk: overall delinquency history,
utilization bands, income-to-debt style ratios, and dependents/credit-line ratios.
This section is written defensively — it only creates a feature if the source columns exist,
so it adapts to whichever Kaggle credit dataset you loaded.


In [ ]:
def safe_col(name):
    return name if name in df.columns else None

# Common "Give Me Some Credit" column names
c_util   = safe_col('RevolvingUtilizationOfUnsecuredLines')
c_age    = safe_col('age')
c_debt   = safe_col('DebtRatio')
c_income = safe_col('MonthlyIncome')
c_lines  = safe_col('NumberOfOpenCreditLinesAndLoans')
c_dep    = safe_col('NumberOfDependents')
c_late30 = safe_col('NumberOfTime30-59DaysPastDueNotWorse')
c_late60 = safe_col('NumberOfTime60-89DaysPastDueNotWorse')
c_late90 = safe_col('NumberOfTimes90DaysLate')
c_re     = safe_col('NumberRealEstateLoansOrLines')

fe = df.copy()

# Total historical delinquencies -- a strong risk signal
late_cols = [c for c in [c_late30, c_late60, c_late90] if c]
if late_cols:
    fe['TotalTimesLate'] = fe[late_cols].sum(axis=1)
    fe['HasEverBeenLate'] = (fe['TotalTimesLate'] > 0).astype(int)

# Income-based ratios
if c_income and c_debt:
    fe['EstimatedMonthlyDebt'] = fe[c_debt] * fe[c_income].replace(0, np.nan)
if c_income and c_dep:
    fe['IncomePerDependent'] = fe[c_income] / (fe[c_dep] + 1)

# Credit line / real-estate mix
if c_lines and c_re:
    fe['NonRealEstateLines'] = fe[c_lines] - fe[c_re]

# Utilization bands (behavior tends to be nonlinear here)
if c_util:
    fe['HighUtilization'] = (fe[c_util] > 1.0).astype(int)  # data quirk: some rows exceed 1.0

print(f"Feature set grew from {df.shape[1]} to {fe.shape[1]} columns")
fe.head()


## 6. Preprocessing

- Impute missing values (median for numeric — robust to the outliers common in income/debt data)
- Scale features for Logistic Regression (tree models don't strictly need this, but it doesn't hurt)


In [ ]:
X = fe.drop(columns=[TARGET_COL])
y = fe[TARGET_COL]

# Keep numeric columns only for this baseline pipeline
X = X.select_dtypes(include=[np.number])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

imputer = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_imp), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imp), columns=X_test.columns, index=X_test.index)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train class balance:\n{y_train.value_counts(normalize=True)}")


## 7. Model Training

Train three classifiers:
- **Logistic Regression** (interpretable baseline, uses scaled features)
- **Decision Tree** (captures nonlinear splits, uses raw imputed features)
- **Random Forest** (ensemble, usually the strongest baseline)

`class_weight='balanced'` is used throughout since credit-default datasets are typically imbalanced.


In [ ]:
models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
                             X_train_scaled, X_test_scaled),
    'Decision Tree': (DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=RANDOM_STATE),
                       X_train_imp, X_test_imp),
    'Random Forest': (RandomForestClassifier(n_estimators=300, max_depth=10, class_weight='balanced',
                                              random_state=RANDOM_STATE, n_jobs=-1),
                       X_train_imp, X_test_imp),
}

trained = {}
for name, (model, Xtr, Xte) in models.items():
    model.fit(Xtr, y_train)
    trained[name] = (model, Xte)
    print(f"Trained: {name}")


## 8. Model Evaluation

Compare models on Accuracy, Precision, Recall, F1-Score, and ROC-AUC.
For credit scoring, **Recall on the default class** and **ROC-AUC** usually matter most —
missing an actual defaulter (false negative) is typically costlier than a false positive.


In [ ]:
results = []
roc_data = {}

for name, (model, Xte) in trained.items():
    y_pred = model.predict(Xte)
    y_proba = model.predict_proba(Xte)[:, 1]

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba),
    })

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_data[name] = (fpr, tpr)

results_df = pd.DataFrame(results).set_index('Model').sort_values('ROC-AUC', ascending=False)
results_df.round(4)


In [ ]:
plt.figure(figsize=(7, 6))
for name, (fpr, tpr) in roc_data.items():
    auc = results_df.loc[name, 'ROC-AUC']
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.show()


In [ ]:
best_model_name = results_df['ROC-AUC'].idxmax()
best_model, best_Xte = trained[best_model_name]
print(f"Best model by ROC-AUC: {best_model_name}")

y_pred_best = best_model.predict(best_Xte)
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'])
plt.title(f'Confusion Matrix - {best_model_name}')
plt.show()

print(classification_report(y_test, y_pred_best))


In [ ]:
# Feature importance (tree-based models only)
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=best_Xte.columns)
    importances = importances.sort_values(ascending=False).head(15)

    plt.figure(figsize=(8, 6))
    importances.sort_values().plot(kind='barh')
    plt.title(f'Top Feature Importances - {best_model_name}')
    plt.xlabel('Importance')
    plt.show()
elif hasattr(best_model, 'coef_'):
    coefs = pd.Series(best_model.coef_[0], index=best_Xte.columns)
    coefs = coefs.sort_values()

    plt.figure(figsize=(8, 6))
    coefs.plot(kind='barh')
    plt.title(f'Logistic Regression Coefficients - {best_model_name}')
    plt.xlabel('Coefficient (standardized features)')
    plt.show()


## 9. Save the Best Model

Persist the trained model (plus the imputer/scaler if needed) so it can be reused for inference
without retraining.


In [ ]:
import os
os.makedirs('../models', exist_ok=True)

joblib.dump(best_model, f'../models/best_model_{best_model_name.replace(" ", "_").lower()}.pkl')
joblib.dump(imputer, '../models/imputer.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

print(f"Saved best model ({best_model_name}) and preprocessing objects to ../models/")


## 10. Summary

- Compared **Logistic Regression**, **Decision Tree**, and **Random Forest** on a credit-scoring
  classification task.
- Evaluated with Precision, Recall, F1-Score, and ROC-AUC to account for class imbalance between
  good-credit and high-risk borrowers.
- Engineered features from payment-history, income, and credit-line data to capture risk signal
  beyond the raw columns.
- Saved the best-performing model for reuse.

**Possible next steps:** hyperparameter tuning with `GridSearchCV`, trying gradient boosting
(XGBoost / LightGBM), SHAP-based explainability, or calibrating predicted probabilities for use
as an actual credit score.
